## Hipotesis 1: PIB per capita y mortalidad por COVID-19

A mayor desarrollo economico de un pais (PIB per capita), menor deberia ser su tasa de mortalidad acumulada por COVID-19.

- **H0:** No existe correlacion significativa entre el PIB per capita y la mortalidad acumulada por millon de habitantes (rho = 0).
- **H1:** Existe correlacion significativa entre ambas variables (rho distinto de 0).
- **Importancia:** permite evaluar si la capacidad economica de un pais se tradujo en mejor proteccion de su poblacion.
- **Variables:** gdp_per_capita y total_deaths_per_million.

In [2]:
import pandas as pd
from scipy import stats

url = "https://raw.githubusercontent.com/owid/covid-19-data/master/public/data/owid-covid-data.csv"
df = pd.read_csv(url)

# Solo paises reales, sin agregados regionales (World, Asia, etc.)
df_paises = df[df['continent'].notna()].copy()

# Snapshot mas reciente por pais
latest = df_paises.sort_values('date').groupby('location').last().reset_index()

# Test H1: PIB per capita vs mortalidad
h1 = latest[['gdp_per_capita', 'total_deaths_per_million']].dropna()
corr1, p1 = stats.spearmanr(h1['gdp_per_capita'], h1['total_deaths_per_million'])
print(f"N paises: {len(h1)}")
print(f"Correlacion de Spearman: r={corr1:.3f}, p-value={p1:.5f}")

N paises: 193
Correlacion de Spearman: r=0.647, p-value=0.00000


**Resultado:** r = 0.647, p < 0.001 -> se rechaza H0, la correlacion es significativa. Pero es positiva, lo opuesto a lo esperado: a mayor PIB per capita, mayor mortalidad reportada. Posible explicacion: subregistro de muertes en paises de menores ingresos por sistemas de vigilancia mas debiles, no necesariamente que los paises ricos sufrieron mas en terminos reales.

## Hipotesis 2: Cobertura de vacunacion y mortalidad

Los paises con mayor cobertura de vacunacion completa presentan menor mortalidad acumulada por COVID-19.

- **H0:** No existe diferencia/correlacion significativa en la mortalidad entre paises con alta y baja cobertura de vacunacion.
- **H1:** Si existe diferencia/correlacion significativa.
- **Importancia:** evalua si las campanas de vacunacion cumplieron su objetivo de reducir muertes.
- **Variables:** people_fully_vaccinated_per_hundred y total_deaths_per_million.

In [3]:
# Test H2: vacunacion vs mortalidad
h2 = latest[['people_fully_vaccinated_per_hundred', 'total_deaths_per_million']].dropna()
corr2, p2 = stats.spearmanr(h2['people_fully_vaccinated_per_hundred'], h2['total_deaths_per_million'])
print(f"N paises: {len(h2)}")
print(f"Correlacion de Spearman: r={corr2:.3f}, p-value={p2:.5f}")

# Comparacion de grupos: alta vs baja vacunacion (split por mediana)
mediana = h2['people_fully_vaccinated_per_hundred'].median()
alta = h2[h2['people_fully_vaccinated_per_hundred'] >= mediana]['total_deaths_per_million']
baja = h2[h2['people_fully_vaccinated_per_hundred'] < mediana]['total_deaths_per_million']
t_stat, p_ttest = stats.ttest_ind(alta, baja, equal_var=False)
print(f"\nAlta vacunacion: n={len(alta)}, media={alta.mean():.1f}")
print(f"Baja vacunacion: n={len(baja)}, media={baja.mean():.1f}")
print(f"T-test: t={t_stat:.3f}, p-value={p_ttest:.5f}")

N paises: 213
Correlacion de Spearman: r=0.213, p-value=0.00178

Alta vacunacion: n=107, media=1379.9
Baja vacunacion: n=106, media=1163.7
T-test: t=1.157, p-value=0.24859


**Resultado:** Spearman r = 0.213, p = 0.0018 (significativo pero debil, y en direccion contraria a la esperada). T-test entre grupos de alta y baja vacunacion: p = 0.249 (ojito no significativo, no se rechaza H0 en esta prueba).

**Conclusion:** los resultados son contraintuitivos pero explicables: probable causalidad inversa (paises mas golpeados por la pandemia aceleraron mas sus campanas de vacunacion) y falta de control por variables de tiempo y edad poblacional. Este hallazgo es clave para la discusion de limitaciones de la Fase 6.